# HDFS Raw Dataset Exploratory Analysis

This notebook analyzes `artifacts/datasets/HDFS/raw/HDFS_raw.csv`, the merged HDFS raw log dataset containing raw log metadata plus `EventId` and `EventTemplate` from the structured parser.

The file is very large, so the full-dataset analysis below streams the CSV in chunks and caches compact summary tables under `artifacts/datasets/HDFS/analysis/`.

In [ ]:
from __future__ import annotations

import json
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_colwidth", 160)
pd.set_option("display.max_rows", 30)


def find_project_root(start: Path) -> Path:
    candidates = [start, *start.parents]
    for candidate in candidates:
        if (candidate / "experiments/behavior_log/artifacts/datasets/HDFS/raw/HDFS_raw.csv").exists():
            return candidate
        nested = candidate / "log-to-vec-benchmark"
        if (nested / "experiments/behavior_log/artifacts/datasets/HDFS/raw/HDFS_raw.csv").exists():
            return nested
    raise FileNotFoundError("Could not find log-to-vec-benchmark project root from the current working directory.")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
DATA_PATH = PROJECT_ROOT / "experiments/behavior_log/artifacts/datasets/HDFS/raw/HDFS_raw.csv"
ANALYSIS_DIR = PROJECT_ROOT / "experiments/behavior_log/artifacts/datasets/HDFS/analysis"
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)

ANALYSIS_PROFILE = "lite"
SAMPLE_ROWS = 20_000
CROSS_SAMPLE_ROWS = 300_000
FULL_SCAN_CHUNKSIZE = 250_000
FULL_SCAN_INCLUDE_CONTENT = False
RUN_FULL_SCAN = False

SUMMARY_JSON = ANALYSIS_DIR / f"HDFS_raw_summary_{ANALYSIS_PROFILE}.json"
EVENT_COUNTS_CSV = ANALYSIS_DIR / f"HDFS_raw_event_counts_{ANALYSIS_PROFILE}.csv"
TEMPLATE_COUNTS_CSV = ANALYSIS_DIR / f"HDFS_raw_template_counts_{ANALYSIS_PROFILE}.csv"
LEVEL_COUNTS_CSV = ANALYSIS_DIR / f"HDFS_raw_level_counts_{ANALYSIS_PROFILE}.csv"
COMPONENT_COUNTS_CSV = ANALYSIS_DIR / f"HDFS_raw_component_counts_{ANALYSIS_PROFILE}.csv"
PID_COUNTS_CSV = ANALYSIS_DIR / f"HDFS_raw_pid_counts_{ANALYSIS_PROFILE}.csv"
DAILY_COUNTS_CSV = ANALYSIS_DIR / f"HDFS_raw_daily_counts_{ANALYSIS_PROFILE}.csv"
HOURLY_COUNTS_CSV = ANALYSIS_DIR / f"HDFS_raw_hourly_counts_{ANALYSIS_PROFILE}.csv"

DATA_PATH

## Quick Preview

Start with a sample to inspect columns, dtypes, and typical values.

In [ ]:
sample = pd.read_csv(DATA_PATH, nrows=SAMPLE_ROWS, low_memory=False)
sample.head()

In [ ]:
sample.info(memory_usage="deep")

In [ ]:
sample.describe(include="all").T

## Full Dataset Aggregation

The notebook defaults to a lite profile for faster startup on the 2.1 GB CSV.

- Full scan: reads only the columns needed for event, template, component, pid, and time analysis.
- Skipped by default: full-dataset `Content` parsing, because it is one of the most expensive parts of the read.
- If you need exact full-dataset content-length stats, set `FULL_SCAN_INCLUDE_CONTENT = True` and rerun the aggregation cell.

In [ ]:
def _counter_to_frame(counter: Counter, key_name: str) -> pd.DataFrame:
    frame = pd.DataFrame(counter.most_common(), columns=[key_name, "count"])
    if len(frame):
        frame["share"] = frame["count"] / frame["count"].sum()
        frame.insert(0, "rank", range(1, len(frame) + 1))
    return frame


def summarize_length_counter(length_counter: Counter[int], total_count: int) -> dict[str, float | int | str | None]:
    if total_count == 0:
        return {"mode": "empty", "count": 0, "mean": 0.0, "min": None, "p50": None, "p90": None, "p95": None, "p99": None, "max": None}

    ordered = sorted(length_counter.items())
    thresholds = {
        "p50": total_count * 0.50,
        "p90": total_count * 0.90,
        "p95": total_count * 0.95,
        "p99": total_count * 0.99,
    }
    percentile_values = {}
    cumulative = 0
    threshold_items = iter(thresholds.items())
    current_name, current_threshold = next(threshold_items)

    for length, count in ordered:
        cumulative += count
        while cumulative >= current_threshold:
            percentile_values[current_name] = int(length)
            try:
                current_name, current_threshold = next(threshold_items)
            except StopIteration:
                current_name = None
                break
        if current_name is None:
            break

    return {
        "mode": "full_scan",
        "count": int(total_count),
        "mean": float(sum(length * count for length, count in length_counter.items()) / total_count),
        "min": int(min(length_counter)),
        "p50": percentile_values.get("p50"),
        "p90": percentile_values.get("p90"),
        "p95": percentile_values.get("p95"),
        "p99": percentile_values.get("p99"),
        "max": int(max(length_counter)),
    }


def build_or_load_summary(force: bool = False, chunksize: int = FULL_SCAN_CHUNKSIZE, include_content: bool = FULL_SCAN_INCLUDE_CONTENT):
    cache_files = [
        SUMMARY_JSON,
        EVENT_COUNTS_CSV,
        TEMPLATE_COUNTS_CSV,
        LEVEL_COUNTS_CSV,
        COMPONENT_COUNTS_CSV,
        PID_COUNTS_CSV,
        DAILY_COUNTS_CSV,
        HOURLY_COUNTS_CSV,
    ]
    if not force and all(path.exists() for path in cache_files):
        summary = json.loads(SUMMARY_JSON.read_text(encoding="utf-8"))
        tables = {
            "event_counts": pd.read_csv(EVENT_COUNTS_CSV),
            "template_counts": pd.read_csv(TEMPLATE_COUNTS_CSV),
            "level_counts": pd.read_csv(LEVEL_COUNTS_CSV),
            "component_counts": pd.read_csv(COMPONENT_COUNTS_CSV),
            "pid_counts": pd.read_csv(PID_COUNTS_CSV),
            "daily_counts": pd.read_csv(DAILY_COUNTS_CSV, parse_dates=["date"]),
            "hourly_counts": pd.read_csv(HOURLY_COUNTS_CSV, parse_dates=["hour"]),
        }
        return summary, tables

    event_counts = Counter()
    template_counts = Counter()
    level_counts = Counter()
    component_counts = Counter()
    pid_counts = Counter()
    daily_counts = Counter()
    hourly_counts = Counter()
    content_length_counts = Counter()
    template_length_counts = Counter()

    total_rows = 0
    missing_counts = Counter()
    min_line_id = None
    max_line_id = None

    usecols = [
        "LineId",
        "Date",
        "Time",
        "Pid",
        "Level",
        "Component",
        "EventId",
        "EventTemplate",
    ]
    if include_content:
        usecols.insert(6, "Content")
    dtypes = {
        "LineId": "int64",
        "Date": "string",
        "Time": "string",
        "Pid": "category",
        "Level": "category",
        "Component": "category",
        "EventId": "category",
        "EventTemplate": "string",
    }
    if include_content:
        dtypes["Content"] = "string"

    for i, chunk in enumerate(
        pd.read_csv(
            DATA_PATH,
            usecols=usecols,
            dtype=dtypes,
            chunksize=chunksize,
            low_memory=False,
            memory_map=True,
        ),
        start=1,
    ):
        total_rows += len(chunk)
        print(f"chunk {i}: total rows = {total_rows:,}")

        event_counts.update(chunk["EventId"].fillna("<missing>").tolist())
        template_counts.update(chunk["EventTemplate"].fillna("<missing>").tolist())
        level_counts.update(chunk["Level"].fillna("<missing>").tolist())
        component_counts.update(chunk["Component"].fillna("<missing>").tolist())
        pid_counts.update(chunk["Pid"].fillna("<missing>").tolist())

        chunk_datetime = pd.to_datetime(
            chunk["Date"].fillna("") + chunk["Time"].fillna(""),
            format="%y%m%d%H%M%S",
            errors="coerce",
        )
        daily_counts.update(chunk_datetime.dt.date.dropna().astype(str).tolist())
        hourly_counts.update(chunk_datetime.dt.floor("h").dropna().astype(str).tolist())

        template_length_counts.update(chunk["EventTemplate"].fillna("").str.count(r"\S+").astype("int16").tolist())
        if include_content:
            content_length_counts.update(chunk["Content"].fillna("").str.count(r"\S+").astype("int16").tolist())

        missing_counts.update(chunk.isna().sum().to_dict())
        chunk_min_line_id = int(chunk["LineId"].min())
        chunk_max_line_id = int(chunk["LineId"].max())
        min_line_id = chunk_min_line_id if min_line_id is None else min(min_line_id, chunk_min_line_id)
        max_line_id = chunk_max_line_id if max_line_id is None else max(max_line_id, chunk_max_line_id)

    event_df = _counter_to_frame(event_counts, "event_id")
    template_df = _counter_to_frame(template_counts, "event_template")
    level_df = _counter_to_frame(level_counts, "level")
    component_df = _counter_to_frame(component_counts, "component")
    pid_df = _counter_to_frame(pid_counts, "pid")

    daily_keys = sorted(daily_counts)
    daily_df = pd.DataFrame(
        {
            "date": pd.to_datetime(daily_keys),
            "total_count": [daily_counts[d] for d in daily_keys],
        }
    )

    hourly_keys = sorted(hourly_counts)
    hourly_df = pd.DataFrame(
        {
            "hour": pd.to_datetime(hourly_keys),
            "total_count": [hourly_counts[h] for h in hourly_keys],
        }
    )

    sample_content_lengths = sample["Content"].fillna("").str.count(r"\S+")
    summary = {
        "input_path": str(DATA_PATH),
        "analysis_profile": ANALYSIS_PROFILE,
        "full_scan_include_content": include_content,
        "total_rows": int(total_rows),
        "unique_events": int(len(event_counts)),
        "unique_templates": int(len(template_counts)),
        "unique_components": int(len(component_counts)),
        "unique_levels": int(len(level_counts)),
        "unique_pids": int(len(pid_counts)),
        "line_id_min": min_line_id,
        "line_id_max": max_line_id,
        "missing_counts": {k: int(v) for k, v in missing_counts.items()},
        "content_word_length": summarize_length_counter(content_length_counts, total_rows) if include_content else {
            "mode": f"sample_{SAMPLE_ROWS}",
            **sample_content_lengths.describe(percentiles=[0.5, 0.9, 0.95, 0.99]).to_dict(),
        },
        "template_word_length": summarize_length_counter(template_length_counts, total_rows),
        "event_top_share": {
            f"top_{k}": float(event_df.head(k)["count"].sum() / total_rows)
            for k in [1, 5, 10, 20, 50, 100]
        },
        "rare_events": {
            "singletons": int((event_df["count"] == 1).sum()),
            "lt_10": int((event_df["count"] < 10).sum()),
            "lt_100": int((event_df["count"] < 100).sum()),
        },
    }

    SUMMARY_JSON.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")
    event_df.to_csv(EVENT_COUNTS_CSV, index=False)
    template_df.to_csv(TEMPLATE_COUNTS_CSV, index=False)
    level_df.to_csv(LEVEL_COUNTS_CSV, index=False)
    component_df.to_csv(COMPONENT_COUNTS_CSV, index=False)
    pid_df.to_csv(PID_COUNTS_CSV, index=False)
    daily_df.to_csv(DAILY_COUNTS_CSV, index=False)
    hourly_df.to_csv(HOURLY_COUNTS_CSV, index=False)

    tables = {
        "event_counts": event_df,
        "template_counts": template_df,
        "level_counts": level_df,
        "component_counts": component_df,
        "pid_counts": pid_df,
        "daily_counts": daily_df,
        "hourly_counts": hourly_df,
    }
    return summary, tables


cache_files = [
    SUMMARY_JSON,
    EVENT_COUNTS_CSV,
    TEMPLATE_COUNTS_CSV,
    LEVEL_COUNTS_CSV,
    COMPONENT_COUNTS_CSV,
    PID_COUNTS_CSV,
    DAILY_COUNTS_CSV,
    HOURLY_COUNTS_CSV,
]

if all(path.exists() for path in cache_files):
    summary, tables = build_or_load_summary(force=False)
    summary
elif RUN_FULL_SCAN:
    summary, tables = build_or_load_summary(force=True)
    summary
else:
    summary, tables = None, None
    print(
        "No cached full-dataset summary found. "
        "Sample-based cells will still work. "
        "Set RUN_FULL_SCAN = True and rerun this cell when you want to build the cached summaries."
    )

## Event Distribution

In [ ]:
event_counts = tables["event_counts"]
event_counts.head(20)

In [ ]:
top_n = 30
plot_df = event_counts.head(top_n).sort_values("count")

fig, ax = plt.subplots(figsize=(10, 9))
sns.barplot(data=plot_df, x="count", y="event_id", ax=ax, color="#4C78A8")
ax.set_title(f"Top {top_n} EventId Frequency")
ax.set_xlabel("Count")
ax.set_ylabel("EventId")
ax.xaxis.set_major_formatter(lambda x, _: f"{x/1_000_000:.1f}M" if x >= 1_000_000 else f"{x/1_000:.0f}K")
plt.tight_layout()

In [ ]:
event_counts = event_counts.copy()
event_counts["cumulative_share"] = event_counts["share"].cumsum()

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(event_counts["rank"], event_counts["cumulative_share"], color="#F58518", linewidth=2)
ax.set_xscale("log")
ax.set_ylim(0, 1.02)
ax.set_title("EventId Cumulative Coverage")
ax.set_xlabel("Event rank, log scale")
ax.set_ylabel("Cumulative share of log lines")
ax.yaxis.set_major_formatter(lambda y, _: f"{y:.0%}")
plt.tight_layout()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
sns.histplot(event_counts["count"], bins=60, log_scale=(True, True), ax=ax, color="#54A24B")
ax.set_title("EventId Frequency Distribution")
ax.set_xlabel("Log lines per EventId, log scale")
ax.set_ylabel("Number of EventIds, log scale")
plt.tight_layout()

## Template Distribution

In [ ]:
template_counts = tables["template_counts"]
template_counts.head(20)

In [ ]:
top_templates = template_counts.head(15).sort_values("count")

fig, ax = plt.subplots(figsize=(11, 8))
sns.barplot(data=top_templates, x="count", y="event_template", ax=ax, color="#B279A2")
ax.set_title("Top 15 Event Templates")
ax.set_xlabel("Count")
ax.set_ylabel("EventTemplate")
plt.tight_layout()

## Severity, Component, and Process Distributions

In [ ]:
display(tables["level_counts"])
display(tables["component_counts"].head(20))
display(tables["pid_counts"].head(20))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.barplot(data=tables["level_counts"].sort_values("count"), x="count", y="level", ax=axes[0], color="#E45756")
axes[0].set_title("Level Distribution")
axes[0].set_xlabel("Count")
axes[0].set_ylabel("Level")

component_plot = tables["component_counts"].head(15).sort_values("count")
sns.barplot(data=component_plot, x="count", y="component", ax=axes[1], color="#72B7B2")
axes[1].set_title("Top 15 Components")
axes[1].set_xlabel("Count")
axes[1].set_ylabel("Component")

plt.tight_layout()

In [ ]:
pid_plot = tables["pid_counts"].head(25).sort_values("count")

fig, ax = plt.subplots(figsize=(10, 8))
sns.barplot(data=pid_plot, x="count", y="pid", ax=ax, color="#FF9DA6")
ax.set_title("Top 25 Process IDs")
ax.set_xlabel("Count")
ax.set_ylabel("Pid")
plt.tight_layout()

## Time Trend

In [ ]:
daily_counts = tables["daily_counts"].sort_values("date")
hourly_counts = tables["hourly_counts"].sort_values("hour")

display(daily_counts.head())
display(hourly_counts.head())

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(13, 8), sharex=False)

axes[0].plot(daily_counts["date"], daily_counts["total_count"], color="#4C78A8", linewidth=1.8)
axes[0].set_title("Daily Log Volume")
axes[0].set_ylabel("Log lines")

axes[1].plot(hourly_counts["hour"], hourly_counts["total_count"], color="#E45756", linewidth=1.4)
axes[1].set_title("Hourly Log Volume")
axes[1].set_xlabel("Time")
axes[1].set_ylabel("Log lines")

plt.tight_layout()

## Message Lengths

In [ ]:
pd.DataFrame(
    {
        "content_word_length": summary["content_word_length"],
        "template_word_length": summary["template_word_length"],
    }
)

In [ ]:
sample_lengths = sample.assign(
    content_word_length=sample["Content"].fillna("").str.split().str.len(),
    template_word_length=sample["EventTemplate"].fillna("").str.split().str.len(),
)

fig, ax = plt.subplots(figsize=(10, 5))
sns.histplot(sample_lengths["content_word_length"], bins=50, label="Content", alpha=0.55, ax=ax)
sns.histplot(sample_lengths["template_word_length"], bins=50, label="EventTemplate", alpha=0.55, ax=ax)
ax.set_title("Word Length Distribution on 20k-Row Sample")
ax.set_xlabel("Word count")
ax.legend()
plt.tight_layout()

## Component-Event Cross Analysis

This section uses a bounded sample by default so the notebook stays responsive.

In [ ]:
cross_sample = pd.read_csv(
    DATA_PATH,
    usecols=["Level", "Component", "EventId", "EventTemplate", "Pid"],
    nrows=CROSS_SAMPLE_ROWS,
    dtype="string",
    low_memory=False,
)

component_event = (
    cross_sample.groupby(["Component", "EventId"], dropna=False)
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)
component_event.head(20)

In [ ]:
component_event_plot = component_event.head(20).copy()
component_event_plot["component_event"] = component_event_plot["Component"].fillna("<missing>") + " | " + component_event_plot["EventId"].fillna("<missing>")
component_event_plot = component_event_plot.sort_values("count")

fig, ax = plt.subplots(figsize=(11, 8))
sns.barplot(data=component_event_plot, x="count", y="component_event", ax=ax, color="#9C755F")
ax.set_title("Top Component-Event Pairs in 1M-Row Sample")
ax.set_xlabel("Count")
ax.set_ylabel("Component | EventId")
plt.tight_layout()

## Notes for Next Steps

- Default aggregation runs in a lite profile, which skips full-dataset `Content` parsing to reduce I/O and memory pressure.
- Use `event_counts` and `template_counts` to decide frequency cutoffs for rare-event handling.
- Use `daily_counts` and `hourly_counts` to check whether time-aware splits are needed.
- Compare `EventId` features with `(Component, EventId)` and `(Level, EventId)` features for downstream representations.
- This merged raw HDFS dataset does not include anomaly labels, so supervised anomaly analysis needs block-level labels from the original HDFS benchmark metadata.